In [1]:
import pandas as pd
from utils import load_disaster_dataset
from snorkel.labeling import labeling_function, LFAnalysis, PandasLFApplier
from snorkel.labeling.model import LabelModel
import re

# 1. Load data
df_train, df_valid, df_test, df_train_true = load_disaster_dataset()

# 2. Define labels
ABSTAIN = -1
NOT_DISASTER = 0
REAL_DISASTER = 1

# 3. Write rules (Labeling Functions)
@labeling_function()
def lf_emergency_keywords(x):
    keywords = ["earthquake", "wildfire", "evacuate", "casualty", "bombing"]
    return REAL_DISASTER if any(word in x.text.lower() for word in keywords) else ABSTAIN

@labeling_function()
def lf_metaphorical(x):
    keywords = ["movie", "song", "metaphor", "literally", "trailer"]
    return NOT_DISASTER if any(word in x.text.lower() for word in keywords) else ABSTAIN

# 🚨 NEW: Added a 3rd Labeling Function so Snorkel can tie-break!
@labeling_function()
def lf_first_responders(x):
    keywords = ["police", "ambulance", "fire truck", "paramedic"]
    return REAL_DISASTER if any(word in x.text.lower() for word in keywords) else ABSTAIN

# Make sure all three are in this list:
lfs = [lf_emergency_keywords, lf_metaphorical, lf_first_responders]
# 4. Apply rules
applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=df_train)

# 5. Train Snorkel to combine the rules intelligently
label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(L_train=L_train, n_epochs=500, log_freq=100, seed=123)

df_train["label"] = label_model.predict(L=L_train, tie_break_policy="abstain")
print("Data successfully labeled by Snorkel!")

100%|██████████| 6090/6090 [00:00<00:00, 18591.89it/s]
INFO:root:Computing O...
INFO:root:Estimating \mu...
  0%|          | 1/500 [00:00<01:41,  4.92epoch/s]INFO:root:[100 epochs]: TRAIN:[loss=0.000]
INFO:root:[200 epochs]: TRAIN:[loss=0.000]
INFO:root:[300 epochs]: TRAIN:[loss=0.000]
INFO:root:[400 epochs]: TRAIN:[loss=0.000]
100%|██████████| 500/500 [00:00<00:00, 1751.05epoch/s]
INFO:root:Finished Training


Data successfully labeled by Snorkel!
